In [0]:
import pandas as pd

files = [
{"file":"map_cities"},
{"file":"map_cancellation_reasons.json"},
{"file":"map_payment_methods.json"},
{"file":"map_vehicle_makes.json"},
{"file":"map_ride_statuses.json"},
{"file":"map_vehicle_types.json"}
]

SAS_TOKEN = spark.conf.get("sas_token")

for file in files:
    name = file["file"].removesuffix(".json")
    url = f"https://uberadls.blob.core.windows.net/raw/Landing/{name}.json?{SAS_TOKEN}"

    df = pd.read_json(url)
    df_spark = spark.createDataFrame(df)
    df_spark.write.format("delta")\
        .mode("overwrite")\
        .option("mergeSchema", True)\
        .saveAsTable(f"uber.bronze.{name}")


In [0]:
SAS_TOKEN = spark.conf.get("sas_token")
url = f"https://uberadls.blob.core.windows.net/raw/Landing/bulk_rides.json?{SAS_TOKEN}"

df = pd.read_json(url)
df_spark = spark.createDataFrame(df)
if not spark.catalog.tableExists("uber.bronze.bulk_rides"):
    df_spark.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(f"uber.bronze.bulk_rides")

In [0]:
%sql
select * from uber.bronze.map_cities